# Avance 2

### Conectarse a la API de Yelp para obtener información de negocios locales según la ciudad seleccionada.

In [59]:
#Importamos las librerías que se necesitan en este avance
import pandas as pd
import requests
#requests: permite enviar solicitudes HTTP a Yelp.
#Pandas nos permite trabajar en dataframes

In [60]:
# La API key está en el archivo API_Key.txt (ignorado por git para no subirla a GitHub)
with open("API_Key.txt", encoding="utf-8") as archivo:
    api_key = archivo.read().strip()

In [61]:
# Ciudad elegida para buscar negocios en Yelp
ciudad = "Chicago"

# Endpoint de Yelp para buscar negocios
url = "https://api.yelp.com/v3/businesses/search"

# Autenticación: enviamos la API key en el encabezado
headers = {"Authorization": f"Bearer {api_key}"}

### Extraer datos como nombre del negocio, categoría, calificación, número de reseñas, calificación, etc. 

In [62]:
#seleccion de 240 restaurantes mediante seis solicitudes de 40 negocios cada una.
negocios = []

for i in range(0, 240, 40):
    params = {"location": ciudad, "categories": "restaurants", "limit": 40, "offset": i}
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()   # detiene el código si Yelp devuelve un error
    negocios.extend(resp.json()["businesses"])
    datos_api = resp.json()

print("Negocios obtenidos:", len(negocios))

Negocios obtenidos: 240


> **Nota:** Yelp no devuelve exactamente los mismos restaurantes en cada consulta, por lo que las cifras
> pueden variar un poco entre ejecuciones. Las conclusiones se redactaron con la ejecución mostrada.

In [63]:
#Veamos los keys del diccionario recibido
datos_api.keys()

dict_keys(['businesses', 'total', 'region'])

In [64]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print(f"Yelp informa {datos_api['total']} resultados para la búsqueda de restaurantes en Chicago.")

Yelp informa 7900 resultados para la búsqueda de restaurantes en Chicago.


In [65]:
#Aqui inspeccionamos el restaurante 
restaurante = datos_api['businesses'][0]
#Esto mostrará los campos disponibles para el primer restaurante.
restaurante.keys()

dict_keys(['id', 'alias', 'name', 'image_url', 'is_closed', 'url', 'review_count', 'categories', 'rating', 'coordinates', 'transactions', 'location', 'phone', 'display_phone', 'distance', 'business_hours', 'attributes'])

In [66]:
columnas = ["nombre", "categoria", "categorias", "rating", "reseñas", "precio"]
datos = []

for negocio in negocios:
    nombre = negocio["name"]
    lista_categorias = [c["title"] for c in negocio["categories"]]
    categoria = lista_categorias[0]             # categoría principal
    categorias = ", ".join(lista_categorias)    # todas las categorías
    rating = negocio["rating"]
    reseñas = negocio["review_count"]
    precio = negocio.get("price")               # puede no existir
    datos.append([nombre, categoria, categorias, rating, reseñas, precio])

df_yelp = pd.DataFrame(datos, columns=columnas)
df_yelp["ciudad"] = ciudad
df_yelp.shape

(240, 7)

In [67]:
df_yelp.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nombre      240 non-null    str    
 1   categoria   240 non-null    str    
 2   categorias  240 non-null    str    
 3   rating      240 non-null    float64
 4   reseñas     240 non-null    int64  
 5   precio      178 non-null    str    
 6   ciudad      240 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 13.3 KB


In [68]:
# Porcentaje de precios faltantes
print(round(df_yelp["precio"].isna().mean() * 100, 1), "%")
#El 25,8% de los restaurantes no tiene dato de precio

25.8 %


In [69]:
# Duplicados: filas completas y nombres repetidos
print("Filas duplicadas:", df_yelp.duplicated().sum())
print("Nombres repetidos:", df_yelp["nombre"].duplicated().sum())
df_yelp[df_yelp["nombre"].duplicated(keep=False)].sort_values("nombre")

Filas duplicadas: 0
Nombres repetidos: 0


,nombre,categoria,categorias,rating,reseñas,precio,ciudad


In [70]:
# Eliminamos filas duplicadas, si las hay (Yelp puede repetir negocios entre consultas)
df_yelp = df_yelp.drop_duplicates().reset_index(drop=True)
print(df_yelp.shape)

# El precio nulo no se imputa: se marca como "No especificado"
df_yelp["precio"] = df_yelp["precio"].fillna("No especificado")
df_yelp.isna().sum()

(240, 7)


nombre        0
categoria     0
categorias    0
rating        0
reseñas       0
precio        0
ciudad        0
dtype: int64

In [71]:
#Revisamos los datos de los dataframe   
print(df_yelp[["rating", "reseñas"]].describe())
print((df_yelp["precio"].value_counts(normalize=True) * 100).round(1))

           rating       reseñas
count  240.000000    240.000000
mean     4.315000    705.900000
std      0.321776   1273.283226
min      2.800000      7.000000
25%      4.200000    144.500000
50%      4.300000    340.500000
75%      4.500000    703.000000
max      5.000000  10765.000000
precio
$$                 49.2
No especificado    25.8
$$$                16.2
$$$$                6.2
$                   2.5
Name: proportion, dtype: float64


#### Exploración de los datos de Yelp
- Se obtuvieron 240 restaurantes de Chicago, sin duplicados.
  Yelp informa 7.900, por lo que la muestra (los más relevantes) no representa toda la ciudad.
- La calificación es alta y homogénea (mediana 4,3; rango 2,8 a 5,0).
- Las reseñas están muy sesgadas (mediana 340,5; máximo 10.765), por lo que se usa la mediana.
- El 25,8% no tiene precio y se marcó como "No especificado". El precio más frecuente es `$$` (49,2%),
  seguido de `$$$` (16,2%); `$$$$` y `$` son pocos (6,2% y 2,5%).

### Integrar esta información al análisis que se está realizando para una ciudad  en específico. 

In [72]:
#Se carga los datos limpios de Chicago y consulta sus dimensiones.
df_chicago = pd.read_csv('C:/Users/USUARIO/Desktop/HENRY/ProyectoM1_Christian_Pacheco/chicago_limpio.csv')
df_chicago.shape

(5384, 21)

In [73]:
# Resumen de Yelp (una fila por ciudad)
resumen_yelp = df_yelp.groupby('ciudad').agg(
    yelp_rating_prom=('rating', 'mean'),
    yelp_reseñas_mediana=('reseñas', 'median'),
    yelp_n_restaurantes=('nombre', 'count')
).round(2).reset_index()

# Unión: se conservan todos los clientes de Chicago
df_integrado = df_chicago.merge(resumen_yelp, left_on='ciudad_residencia', right_on='ciudad', how='left')
print(df_integrado.shape, '| clientes:', df_chicago.shape[0])

(5384, 25) | clientes: 5384


In [74]:
#Aquí se compara las preferencias alimenticias, categorías de Yelp, gasto mediano y distribución de precios.
print('Preferencias de los clientes (%):')
print((df_chicago['preferencias_alimenticias'].value_counts(normalize=True).head(3) * 100).round(1))
print('\nCategorías más frecuentes en Yelp:')
print(df_yelp['categoria'].value_counts().head(5))
print('\nGasto mediano de los clientes:', df_chicago['promedio_gasto_comida'].median())
print('\nPrecio de los restaurantes (%):')
print((df_yelp['precio'].value_counts(normalize=True) * 100).round(1))

Preferencias de los clientes (%):
preferencias_alimenticias
Carnes         29.5
Vegetariano    20.9
Mariscos       12.9
Name: proportion, dtype: float64

Categorías más frecuentes en Yelp:
categoria
New American    14
Italian         12
Mexican         11
Pizza           10
American         9
Name: count, dtype: int64

Gasto mediano de los clientes: 19.175

Precio de los restaurantes (%):
precio
$$                 49.2
No especificado    25.8
$$$                16.2
$$$$                6.2
$                   2.5
Name: proportion, dtype: float64


In [75]:
#Aqui eliminamos la columna ciudad del dataframe integrado, ya que está repetida
df_integrado = df_integrado.drop(columns='ciudad')

### Conclusiones del Avance 2
- Se conectó a la API de Yelp y se extrajeron 240 restaurantes de Chicago (nombre, categoría, calificación,
  reseñas y precio). Yelp informa 7.900, por lo que la muestra no representa toda la ciudad.
- Se unió el resumen de Yelp con los clientes de Chicago por ciudad. Como todos los clientes viven en
  Chicago, las columnas de Yelp tienen el mismo valor en todas las filas.
- **Precio:** el nivel más frecuente es `$$` (49,2%), y el gasto mediano de los clientes (19,18) es
  coherente con una oferta de precio medio.
- **Preferencias:** los clientes prefieren carnes (29,5%) y vegetariano (20,9%), mientras que las
  categorías más comunes en Yelp son tipos de cocina como New American, Italian y Mexican; ninguna es
  vegetariana. Se plantea como hipótesis una demanda vegetariana poco cubierta.
- Las categorías de Yelp no equivalen a las preferencias alimenticias, por lo que la comparación es
  aproximada.